In [14]:
import sys 
sys.path.append('../')
from torch.utils.data import  DataLoader
from chess_engine.src.model.config.config import  model_settings, data_settings
from chess_engine.src.model.classes.npz_piping.dataloader import NpzDataset

In [15]:
train_dataset = NpzDataset(data_settings.TrainingDirectory)

In [16]:
train_loader = DataLoader(train_dataset, batch_size=model_settings.DataLoaderBatchSize, shuffle=True, num_workers=0)

In [17]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class NPZDirectoryDataset(Dataset):
    def __init__(self, directory):
        """
        Initializes the dataset by loading all npz files in the directory.
        
        Args:
            directory (str): Path to the directory containing .npz files.
        """
        # Find all npz files in the directory
        self.files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.npz')]
        
        # Load and concatenate all data
        all_features = []
        all_labels = []
        
        for file in self.files:
            data = np.load(file)
            features = data['features']    # shape: [N, ...]
            labels = data['labels']        # shape: [N]
            all_features.append(features)
            all_labels.append(labels)
        
        # Concatenate all arrays along the first dimension
        self.features = np.concatenate(all_features, axis=0)  # shape: [total_N, ...]
        self.labels = np.concatenate(all_labels, axis=0)      # shape: [total_N]

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        # Convert data to torch tensors
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y





In [18]:
dataset = NPZDirectoryDataset(data_settings.TestingDirectory)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)



In [20]:
# Example iteration
i = 0
for batch_features, batch_labels in dataloader:
    # Your training or evaluation code here
    print(f"features shape: {batch_features.shape} labels shape: {batch_labels.shape}")
    i += 1
    pass

features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])
features shape: torch.Size([32, 12, 8, 8]) labels shape: torch.Size([32, 3])

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.optim as optim
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.tensorboard import SummaryWriter  # For TensorBoard
from chess_engine.src.model.config.config import model_settings, data_settings
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import sample_bitboard_dict
from chess_engine.src.model.classes.npz_piping.dataloader import NpzDataset
from chess_engine.src.model.classes.torch_model import AlphaZeroNet  # Assuming you put AlphaZeroNet in alpha_zero_net.py



# Hyperparameters
batch_size = 64
learning_rate = 1e-3
num_epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create Datasets and DataLoaders
train_dataset = NPZDirectoryDataset(data_settings.TrainingDirectory)
val_dataset = NPZDirectoryDataset(data_settings.ValidationDirectory)
test_dataset = NPZDirectoryDataset(data_settings.TestingDirectory)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Initialize model
model = AlphaZeroNet(n_bitboards=len(sample_bitboard_dict.keys()), board_size=8)
model = model.to(device)

# Define optimizers and loss functions
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Assume the dataset returns `x, (policy_label, value_label)` pairs
# If it returns a single label, you may need to split it accordingly.
# For example, if `labels` contains both move distribution and outcome:
# labels could be a dict or tuple: (policy_targets, value_targets)
# Adjust the dataset or this code depending on how you structured your dataset labels.
policy_loss_fn = nn.NLLLoss()  # policy head outputs log_softmax
value_loss_fn = nn.CrossEntropyLoss()  # value head outputs raw logits (for classes)

# Training loop
for epoch in range(num_epochs):
    model.train()
    train_policy_loss = 0.0
    train_value_loss = 0.0

    # Training
    for batch_features, batch_labels in tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{num_epochs}"):
        batch_features = batch_features.to(device)  # shape: [B, n_bitboards, 8, 8]

        # Assuming batch_labels is a tuple: (policy_targets, value_targets)
        # policy_targets: [B] (class indices or probability distribution)
        # value_targets: [B] (0,1,2 for the value outcome)
        policy_targets, value_targets = batch_labels
        policy_targets = policy_targets.to(device)  # shape: [B]
        value_targets = value_targets.to(device)    # shape: [B]

        optimizer.zero_grad()

        # Forward pass
        policy_pred, value_pred = model(batch_features)
        # policy_pred: [B, board_size * board_size]
        # value_pred: [B, 3]

        # Compute losses
        loss_policy = policy_loss_fn(policy_pred, policy_targets)
        loss_value = value_loss_fn(value_pred, value_targets)
        loss = loss_policy + loss_value

        loss.backward()
        optimizer.step()

        train_policy_loss += loss_policy.item() * batch_features.size(0)
        train_value_loss += loss_value.item() * batch_features.size(0)

    # Compute average losses for the epoch
    avg_policy_loss = train_policy_loss / len(train_dataset)
    avg_value_loss = train_value_loss / len(train_dataset)

    print(f"Epoch {epoch+1}/{num_epochs}, Policy Loss: {avg_policy_loss:.4f}, Value Loss: {avg_value_loss:.4f}")

    # Validation step
    model.eval()
    val_policy_loss = 0.0
    val_value_loss = 0.0
    with torch.no_grad():
        for batch_features, batch_labels in val_loader:
            batch_features = batch_features.to(device)
            policy_targets, value_targets = batch_labels
            policy_targets = policy_targets.to(device)
            value_targets = value_targets.to(device)

            policy_pred, value_pred = model(batch_features)

            loss_policy = policy_loss_fn(policy_pred, policy_targets)
            loss_value = value_loss_fn(value_pred, value_targets)

            val_policy_loss += loss_policy.item() * batch_features.size(0)
            val_value_loss += loss_value.item() * batch_features.size(0)

    avg_val_policy_loss = val_policy_loss / len(val_dataset)
    avg_val_value_loss = val_value_loss / len(val_dataset)
    print(f"Validation: Policy Loss: {avg_val_policy_loss:.4f}, Value Loss: {avg_val_value_loss:.4f}")

# After training, you can test the model on the test set
model.eval()
test_policy_loss = 0.0
test_value_loss = 0.0
all_value_preds = []
all_value_targets = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        policy_targets, value_targets = batch_labels
        policy_targets = policy_targets.to(device)
        value_targets = value_targets.to(device)

        policy_pred, value_pred = model(batch_features)

        loss_policy = policy_loss_fn(policy_pred, policy_targets)
        loss_value = value_loss_fn(value_pred, value_targets)

        test_policy_loss += loss_policy.item() * batch_features.size(0)
        test_value_loss += loss_value.item() * batch_features.size(0)

        all_value_preds.append(torch.argmax(value_pred, dim=1).cpu().numpy())
        all_value_targets.append(value_targets.cpu().numpy())

avg_test_policy_loss = test_policy_loss / len(test_dataset)
avg_test_value_loss = test_value_loss / len(test_dataset)
all_value_preds = np.concatenate(all_value_preds)
all_value_targets = np.concatenate(all_value_targets)

print(f"Test Results: Policy Loss: {avg_test_policy_loss:.4f}, Value Loss: {avg_test_value_loss:.4f}")

# Compute some metrics for value predictions as an example
precision = precision_score(all_value_targets, all_value_preds, average='weighted')
recall = recall_score(all_value_targets, all_value_preds, average='weighted')
f1 = f1_score(all_value_targets, all_value_preds, average='weighted')
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")

conf_mat = confusion_matrix(all_value_targets, all_value_preds)
print("Confusion Matrix:")
print(conf_mat)


Training Epoch 1/10:   0%|                                                                                                                                                                                | 0/34849 [00:00<?, ?it/s]


ValueError: too many values to unpack (expected 2)